# 05 - Ingestão Silver: Limpeza, Tratamento e Validação de Regras do Micro-Lote
**Squad 2 — Real Time for Business | Dupla 1**  
**Integrantes:** Lucas Sousa Santos Oliveira & Zaiden Emiliano Segundo Seleme  
**Tabelas de Escopo:** `ecommerce_produtos` e `ecommerce_categorias`  
**Branch:** `feat/squad2-lucas_zaiden`  

### Objetivo da Task (Sprint 2 - Task 2):
1. **Leitura Incremental da Camada Bronze:** Consumir os micro-lotes recém-ingeridos na Bronze utilizando controle de watermark (`bronze_ingested_at`).
2. **Auditoria Silver:** Adicionar a coluna de rastreabilidade `'silver_processed_at'` com `current_timestamp()`.
3. **Limpeza e Padronização:** Aplicar `trim` nos campos textuais e casting estrito de tipos (`preco_lista`, `is_ativo`).
4. **Validação das Regras Técnicas (Padrão Quarentena):**
   * `ecommerce_produtos`:
     * *Técnica 1:* Comprimento do SKU entre 5 e 60 caracteres (`5 < len(sku) < 60`).
     * *Técnica 2:* Faixa operacional de preço (`0 < preco_lista < 5000`).
     * *Técnica 3:* `is_ativo` não nulo e booleano.
   * `ecommerce_categorias`:
     * *Técnica 1:* `id_categoria` e `nome_categoria` obrigatórios (não nulos nem vazios).
   * Registros que violarem as regras são encaminhados para a **Quarentena** com o motivo em `quarantine_reason`.
5. **Monitoramento e Alertas de Negócio:**
   * *Negócio 4 (Produtos):* Alertar se o micro-lote adicionar `> 50` novos SKUs (possível carga de teste).
   * *Negócio 5 (Produtos):* Alertar criticamente se produto ativo tiver `preco_lista <= 0`.
   * *Negócio 2 (Categorias):* Alertar se a contagem de categorias raiz (`id_categoria_pai IS NULL`) for alterada.
6. **Persistência em Delta Silver (`mode: append`):** Salvar os registros válidos no container `squad2/grupo1/silver/` em formato Delta e registrar no Databricks Metastore.

## 1. Carregamento de Credenciais e Configurações de Conexão
Injeção granular de `adls_options` diretamente em cada operação do Spark, garantindo compatibilidade nativa com Databricks Serverless.

In [0]:
from dotenv import load_dotenv
import os

dotenv_path = None
for root_candidate in [
    os.getcwd(),
    os.path.abspath(os.path.join(os.getcwd(), "..")),
    os.path.abspath(os.path.join(os.getcwd(), "../..")),
    "/Workspace/Shared/estagio_ed",
    "/Workspace/Repos/estagio_ed"
]:
    p = os.path.join(root_candidate, ".env")
    if os.path.exists(p):
        dotenv_path = p
        break

if not dotenv_path:
    for dirpath, _, filenames in os.walk(os.getcwd()):
        if ".env" in filenames:
            dotenv_path = os.path.join(dirpath, ".env")
            break

load_dotenv(dotenv_path, override=True)

storage_account = os.getenv("ADLS_STORAGE_ACCOUNT_NAME", "internshipdatalake")
client_id = os.getenv("ADLS_CLIENT_ID")
tenant_id = os.getenv("ADLS_TENANT_ID")
client_secret = os.getenv("ADLS_CLIENT_SECRET")

# Configurações OAuth do Service Principal para injeção granular nas operações do cluster
adls_options = {
    f"fs.azure.account.auth.type.{storage_account}.dfs.core.windows.net": "OAuth",
    f"fs.azure.account.oauth.provider.type.{storage_account}.dfs.core.windows.net": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{storage_account}.dfs.core.windows.net": client_id,
    f"fs.azure.account.oauth2.client.secret.{storage_account}.dfs.core.windows.net": client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{storage_account}.dfs.core.windows.net": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

# Definição dos caminhos com isolamento de namespace no prefixo /grupo1/
base_squad = f"abfss://squad2@{storage_account}.dfs.core.windows.net/grupo1"

caminhos = {
    "bronze_produtos": f"{base_squad}/bronze/ecommerce_produtos",
    "bronze_categorias": f"{base_squad}/bronze/ecommerce_categorias",
    "silver_produtos": f"{base_squad}/silver/ecommerce_produtos",
    "silver_categorias": f"{base_squad}/silver/ecommerce_categorias",
    "quarantine_produtos": f"{base_squad}/quarantine/ecommerce_produtos",
    "quarantine_categorias": f"{base_squad}/quarantine/ecommerce_categorias"
}

# Função auxiliar para verificar existência física de tabelas Delta de forma eager e segura
def tabela_delta_existe(caminho):
    try:
        spark.read.format("delta").options(**adls_options).load(caminho).limit(1).count()
        return True
    except Exception:
        return False

print("Caminhos configurados para a Camada Silver:")
for k, v in caminhos.items():
    print(f"  {k}: {v}")

## 2. Funções de Tratamento, Validação e Quarentena de Micro-Lote
Separa registros válidos dos reprovados e emite alertas operacionais de negócio.

In [0]:
from pyspark.sql.functions import (
    current_timestamp, col, trim, length, when, lit, concat_ws, max as spark_max, countDistinct
)
from pyspark.sql.types import DoubleType, BooleanType

# Processador do lote de Produtos
def processar_silver_produtos(df_lote):
    total = df_lote.count()
    if total == 0:
        print("Nenhum registro novo em Bronze para processar na Silver de 'ecommerce_produtos'.")
        return

    print(f"\n===================================================")
    print(f">>> Processando Silver: ecommerce_produtos ({total} registros novos da Bronze)")

    # 1. Auditoria Silver, Limpeza de Strings e Tipagem Estrita
    df_cleaned = (df_lote
        .withColumn("silver_processed_at", current_timestamp())
        .withColumn("sku", trim(col("sku")))
        .withColumn("nome_produto", trim(col("nome_produto")))
        .withColumn("descricao", trim(col("descricao")))
        .withColumn("id_categoria", trim(col("id_categoria")))
        .withColumn("unidade_medida", trim(col("unidade_medida")))
        .withColumn("nome_marca", trim(col("nome_marca")))
        .withColumn("preco_lista", col("preco_lista").cast(DoubleType()))
        .withColumn("is_ativo", col("is_ativo").cast(BooleanType())))

    # 2. Regras Técnicas (Validações de Quarentena)
    cond_sku_valido = (length(col("sku")) > 5) & (length(col("sku")) < 60)
    cond_preco_valido = (col("preco_lista") > 0) & (col("preco_lista") < 5000)
    cond_ativo_valido = col("is_ativo").isNotNull()

    df_tagged = df_cleaned.withColumn(
        "quarantine_reason",
        concat_ws("; ",
            when(~cond_sku_valido, lit("FALHA_TECNICA_1: Comprimento do SKU fora da faixa (5 a 60 caracteres)")),
            when(~cond_preco_valido, lit("FALHA_TECNICA_2: Preço de lista fora da faixa permitida (0 a 5000)")),
            when(~cond_ativo_valido, lit("FALHA_TECNICA_3: Campo is_ativo nulo ou tipo inválido"))
        )
    ).withColumn("quarantined_at", current_timestamp())

    df_validos = df_tagged.filter(col("quarantine_reason") == "").drop("quarantine_reason", "quarantined_at")
    df_quarentena = df_tagged.filter(col("quarantine_reason") != "")

    total_validos = df_validos.count()
    total_quarentena = df_quarentena.count()
    print(f"    -> Registros Válidos:    {total_validos}")
    print(f"    -> Registros Quarentena: {total_quarentena}")

    # 3. Regras de Negócio e Alertas Operacionais
    anomalias_preco = df_cleaned.filter((col("is_ativo") == True) & ((col("preco_lista") <= 0) | col("preco_lista").isNull())).count()
    if anomalias_preco > 0:
        print(f"    🚨 [ALERTA CRÍTICO DE NEGÓCIO - REGRA 5]: Detectados {anomalias_preco} produtos ATIVOS com preço <= 0!")

    if tabela_delta_existe(caminhos["silver_produtos"]):
        df_silver_existente = spark.read.format("delta").options(**adls_options).load(caminhos["silver_produtos"])
        skus_novos = df_validos.join(df_silver_existente, on="sku", how="left_anti").select(countDistinct("sku")).first()[0]
    else:
        skus_novos = df_validos.select(countDistinct("sku")).first()[0]

    print(f"    📊 [KPI NEGÓCIO - REGRA 4]: Novos SKUs adicionados neste lote: {skus_novos}")
    if skus_novos > 50:
        print(f"    ⚠️ [ALERTA DE NEGÓCIO - REGRA 4]: Lote com mais de 50 novos SKUs ({skus_novos}). Verificar se é carga de teste!")

    # 4. Gravação Delta Silver em modo append (conforme especificação da task)
    if total_validos > 0:
        (df_validos.write
            .format("delta")
            .options(**adls_options)
            .mode("append")
            .save(caminhos["silver_produtos"]))
        print(f"    -> {total_validos} registros gravados com sucesso na Delta Silver.")

    # 5. Gravação da Quarentena
    if total_quarentena > 0:
        (df_quarentena.write
            .format("delta")
            .options(**adls_options)
            .mode("append")
            .save(caminhos["quarantine_produtos"]))
        print(f"    -> {total_quarentena} registros isolados gravados na Quarentena.")

    print(f"<<< Processamento Silver concluído com sucesso.")
    print(f"===================================================")


# Processador do lote de Categorias
def processar_silver_categorias(df_lote):
    total = df_lote.count()
    if total == 0:
        print("Nenhum registro novo em Bronze para processar na Silver de 'ecommerce_categorias'.")
        return

    print(f"\n===================================================")
    print(f">>> Processando Silver: ecommerce_categorias ({total} registros novos da Bronze)")

    # 1. Auditoria Silver e Limpeza de Strings
    df_cleaned = (df_lote
        .withColumn("silver_processed_at", current_timestamp())
        .withColumn("id_categoria", trim(col("id_categoria")))
        .withColumn("nome_categoria", trim(col("nome_categoria")))
        .withColumn("id_categoria_pai", trim(col("id_categoria_pai")))
        .withColumn("nome_categoria_pai", trim(col("nome_categoria_pai")))
        .withColumn("tipo_categoria", trim(col("tipo_categoria"))))

    # 2. Regras Técnicas
    cond_id_valido = col("id_categoria").isNotNull() & (length(col("id_categoria")) > 0)
    cond_nome_valido = col("nome_categoria").isNotNull() & (length(col("nome_categoria")) > 0)

    df_tagged = df_cleaned.withColumn(
        "quarantine_reason",
        concat_ws("; ",
            when(~cond_id_valido, lit("FALHA_TECNICA_1: id_categoria nulo ou vazio")),
            when(~cond_nome_valido, lit("FALHA_TECNICA_1: nome_categoria nulo ou vazio"))
        )
    ).withColumn("quarantined_at", current_timestamp())

    df_validos = df_tagged.filter(col("quarantine_reason") == "").drop("quarantine_reason", "quarantined_at")
    df_quarentena = df_tagged.filter(col("quarantine_reason") != "")

    total_validos = df_validos.count()
    total_quarentena = df_quarentena.count()
    print(f"    -> Registros Válidos:    {total_validos}")
    print(f"    -> Registros Quarentena: {total_quarentena}")

    # 3. Regra de Negócio 2: Categorias raiz
    cat_raiz_lote = df_validos.filter(col("id_categoria_pai").isNull() | (col("id_categoria_pai") == "")).select(countDistinct("id_categoria")).first()[0]
    print(f"    🌳 [GOVERNANÇA NEGÓCIO - REGRA 2]: Categorias raiz identificadas no lote: {cat_raiz_lote}")

    if tabela_delta_existe(caminhos["silver_categorias"]):
        df_silver_cat = spark.read.format("delta").options(**adls_options).load(caminhos["silver_categorias"])
        cat_raiz_existentes = df_silver_cat.filter(col("id_categoria_pai").isNull() | (col("id_categoria_pai") == "")).select(countDistinct("id_categoria")).first()[0]
        if cat_raiz_lote != cat_raiz_existentes and cat_raiz_existentes > 0:
            print(f"    ⚠️ [ALERTA DE NEGÓCIO - REGRA 2]: Número total de categorias raiz mudou! (Anterior: {cat_raiz_existentes} vs Lote: {cat_raiz_lote}). Impacto na navegação do e-commerce!")

    # 4. Gravação Delta Silver (modo append)
    if total_validos > 0:
        (df_validos.write
            .format("delta")
            .options(**adls_options)
            .mode("append")
            .save(caminhos["silver_categorias"]))
        print(f"    -> {total_validos} registros gravados com sucesso na Delta Silver.")

    # 5. Gravação da Quarentena
    if total_quarentena > 0:
        (df_quarentena.write
            .format("delta")
            .options(**adls_options)
            .mode("append")
            .save(caminhos["quarantine_categorias"]))
        print(f"    -> {total_quarentena} registros isolados gravados na Quarentena.")

    print(f"<<< Processamento Silver concluído com sucesso.")
    print(f"===================================================")

print("Processadores Silver configurados com sucesso.")

## 3. Consumo Incremental da Bronze: `ecommerce_produtos`
Lê os registros da Bronze posteriores à última ingestão na Silver (controle via watermark temporal `bronze_ingested_at`).

In [0]:
print("Verificando novos registros em Bronze para 'ecommerce_produtos'...")

if not tabela_delta_existe(caminhos["bronze_produtos"]):
    print("⚠️ Tabela Bronze de 'ecommerce_produtos' ainda não existe no storage. Execute primeiro o notebook 04_bronze_ingestao_delta.")
else:
    df_bronze_prod = spark.read.format("delta").options(**adls_options).load(caminhos["bronze_produtos"])

    if tabela_delta_existe(caminhos["silver_produtos"]):
        df_silver_prod_atual = spark.read.format("delta").options(**adls_options).load(caminhos["silver_produtos"])
        ultimo_processado = df_silver_prod_atual.select(spark_max(col("bronze_ingested_at"))).first()[0]
        if ultimo_processado:
            print(f"Último watermark processado na Silver: {ultimo_processado}")
            df_novos_prod = df_bronze_prod.filter(col("bronze_ingested_at") > ultimo_processado)
        else:
            df_novos_prod = df_bronze_prod
    else:
        # Primeira carga (tabela Silver ainda não existe no storage)
        print("Primeira carga Silver detectada (sem watermark anterior).")
        df_novos_prod = df_bronze_prod

    processar_silver_produtos(df_novos_prod)

## 4. Consumo Incremental da Bronze: `ecommerce_categorias`
Executa o processamento incremental para a tabela de categorias com controle de watermark.

In [0]:
print("Verificando novos registros em Bronze para 'ecommerce_categorias'...")

if not tabela_delta_existe(caminhos["bronze_categorias"]):
    print("⚠️ Tabela Bronze de 'ecommerce_categorias' ainda não existe no storage. Execute primeiro o notebook 04_bronze_ingestao_delta.")
else:
    df_bronze_cat = spark.read.format("delta").options(**adls_options).load(caminhos["bronze_categorias"])

    if tabela_delta_existe(caminhos["silver_categorias"]):
        df_silver_cat_atual = spark.read.format("delta").options(**adls_options).load(caminhos["silver_categorias"])
        ultimo_processado_cat = df_silver_cat_atual.select(spark_max(col("bronze_ingested_at"))).first()[0]
        if ultimo_processado_cat:
            print(f"Último watermark processado na Silver: {ultimo_processado_cat}")
            df_novos_cat = df_bronze_cat.filter(col("bronze_ingested_at") > ultimo_processado_cat)
        else:
            df_novos_cat = df_bronze_cat
    else:
        print("Primeira carga Silver detectada (sem watermark anterior).")
        df_novos_cat = df_bronze_cat

    processar_silver_categorias(df_novos_cat)

## 5. Criação e Mapeamento das Tabelas Externas no Databricks Metastore
Mapeia as tabelas Silver e Quarentena sob o schema `squad2` no catálogo do Databricks.

In [0]:
try:
    spark.sql("CREATE SCHEMA IF NOT EXISTS squad2")

    # 1. Tabela Externa Silver de Produtos
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS squad2.silver_ecommerce_produtos
    USING DELTA
    LOCATION '{caminhos["silver_produtos"]}'
    """)

    # 2. Tabela Externa Silver de Categorias
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS squad2.silver_ecommerce_categorias
    USING DELTA
    LOCATION '{caminhos["silver_categorias"]}'
    """)

    # 3. Tabela Externa de Quarentena de Produtos
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS squad2.quarantine_ecommerce_produtos
    USING DELTA
    LOCATION '{caminhos["quarantine_produtos"]}'
    """)

    # 4. Tabela Externa de Quarentena de Categorias
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS squad2.quarantine_ecommerce_categorias
    USING DELTA
    LOCATION '{caminhos["quarantine_categorias"]}'
    """)
    print("Tabelas externas Silver e Quarentena registradas no Databricks Metastore!")
except Exception as e:
    print(f"Nota sobre Metastore Externo: {e}")
    print("Os dados Delta no ADLS e views temporárias estão prontos para consulta local.")

## 6. Auditoria de Qualidade dos Dados na Camada Silver

In [0]:
print("=== Relatório de Auditoria Silver & Quarentena ===\n")

# 1. Auditoria de Produtos na Silver
if tabela_delta_existe(caminhos["silver_produtos"]):
    df_silver_prod = spark.read.format("delta").options(**adls_options).load(caminhos["silver_produtos"])
    df_silver_prod.createOrReplaceTempView("silver_ecommerce_produtos")
    total_prod = df_silver_prod.count()
    unicos_prod = df_silver_prod.select(countDistinct("sku")).first()[0]
    print(f"Produtos na Silver:   {total_prod} registros ({unicos_prod} SKUs únicos)")
    print("\n--- Amostra Silver: Produtos higienizados com 'silver_processed_at' ---")
    display(df_silver_prod.select("sku", "nome_produto", "preco_lista", "is_ativo", "silver_processed_at").limit(5))
else:
    print("Tabela silver_ecommerce_produtos ainda não existe no storage.")

# 2. Auditoria de Categorias na Silver
if tabela_delta_existe(caminhos["silver_categorias"]):
    df_silver_cat = spark.read.format("delta").options(**adls_options).load(caminhos["silver_categorias"])
    df_silver_cat.createOrReplaceTempView("silver_ecommerce_categorias")
    total_cat = df_silver_cat.count()
    unicos_cat = df_silver_cat.select(countDistinct("id_categoria")).first()[0]
    print(f"\nCategorias na Silver: {total_cat} registros ({unicos_cat} Categorias únicas)")
    print("\n--- Amostra Silver: Categorias higienizadas com 'silver_processed_at' ---")
    display(df_silver_cat.select("id_categoria", "nome_categoria", "tipo_categoria", "silver_processed_at").limit(5))
else:
    print("\nTabela silver_ecommerce_categorias ainda não existe no storage.")

# 3. Verificação de Quarentena
if tabela_delta_existe(caminhos["quarantine_produtos"]):
    df_quar_prod = spark.read.format("delta").options(**adls_options).load(caminhos["quarantine_produtos"])
    df_quar_prod.createOrReplaceTempView("quarantine_ecommerce_produtos")
    print(f"\nProdutos na Quarentena: {df_quar_prod.count()} registros")
    print("\nAmostra de Produtos Reprovados:")
    display(df_quar_prod.select("sku", "preco_lista", "is_ativo", "quarantine_reason").limit(5))
else:
    print("\nProdutos na Quarentena: 0 registros (tabela de quarentena não gerada ou vazia).")

if tabela_delta_existe(caminhos["quarantine_categorias"]):
    df_quar_cat = spark.read.format("delta").options(**adls_options).load(caminhos["quarantine_categorias"])
    df_quar_cat.createOrReplaceTempView("quarantine_ecommerce_categorias")
    print(f"\nCategorias na Quarentena: {df_quar_cat.count()} registros")
else:
    print("\nCategorias na Quarentena: 0 registros (tabela de quarentena não gerada ou vazia).")